In [1]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np

from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score
)

from imblearn.over_sampling import SMOTE


In [2]:
# 2. LOAD DATA

df = pd.read_csv("../data/processed.csv")
print("Shape:", df.shape)
df.head()


Shape: (15420, 52)


,month,weekofmonth,dayofweek,make,accidentarea,dayofweekclaimed,monthclaimed,weekofmonthclaimed,sex,maritalstatus,...,speed_volatility,harsh_braking_risk,harsh_acceleration_risk,harsh_cornering_risk,harsh_driving_index,high_night_driving,high_urban_driving,fast_claim,high_claim_history,claim_driving_risk
0,2,1.717545,6,6,1,6,5,-1.345408,0,2,...,0.725376,-0.301255,-0.524215,-0.134501,-0.340056,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
1,4,0.164199,6,6,1,2,5,1.037295,1,2,...,-0.317649,-0.301255,1.907613,-0.134501,1.661724,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
2,10,1.717545,0,6,1,5,10,-0.551174,1,1,...,1.399306,-0.301255,-0.524215,-0.134501,-0.006426,1.010037,1.139325,-0.008053,-0.972492,-0.008053
3,6,-0.612473,2,17,0,1,6,-1.345408,1,1,...,-0.821335,-0.301255,1.907613,-0.134501,0.994464,-0.990062,1.139325,-0.008053,-0.972492,-0.008053
4,4,1.717545,1,6,1,6,4,-0.551174,0,2,...,0.170197,-0.301255,-0.524215,-0.134501,-1.340946,-0.990062,1.139325,-0.008053,-0.972492,-0.008053


In [3]:
# 3. IDENTIFY TARGET COLUMN

target_col = None
for col in df.columns:
    if 'fraud' in col.lower() or 'label' in col.lower():
        target_col = col
        break

if target_col is None:
    raise ValueError(" Target column not found!")

print(" Target Column:", target_col)


 Target Column: fraudfound_p


In [4]:
# 4. HANDLE MISSING VALUES

df.ffill(inplace=True)


In [5]:
# 5. FEATURE ENGINEERING

engineered_features = []
if 'night_driving_ratio' in df.columns and 'distance_km' in df.columns:
    df['risk_feature'] = df['night_driving_ratio'] * df['distance_km']
    engineered_features.append('risk_feature')

print("Engineered Features:", engineered_features)


Engineered Features: ['risk_feature']


In [6]:
# 6. TRAIN-TEST SPLIT

X = df.drop(columns=[target_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [7]:
# 7. HANDLE CLASS IMBALANCE (SMOTE)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train_res.shape)


After SMOTE: (23196, 52)


In [8]:
# 8. IDENTIFY CATEGORICAL FEATURES

cat_features = X.select_dtypes(include=['object']).columns.tolist()
print("Categorical Features:\n", cat_features)


Categorical Features:
 []


In [10]:
# 9. TRAIN MODEL (CATBOOST)

model = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    loss_function='Logloss',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=100
)
model.fit(
    X_train_res, y_train_res,
    cat_features=cat_features,
    eval_set=(X_test, y_test)
)


0:	test: 0.7938506	best: 0.7938506 (0)	total: 201ms	remaining: 3m 20s
100:	test: 0.8151851	best: 0.8173294 (94)	total: 5.27s	remaining: 46.9s
200:	test: 0.8220635	best: 0.8234638 (171)	total: 7.93s	remaining: 31.5s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8234638226
bestIteration = 171

Shrink model to first 172 iterations.


CatBoostClassifier(depth=6, early_stopping_rounds=50, eval_metric='AUC', iterations=1000, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [11]:
# 10. CROSS VALIDATION

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

for train_idx, val_idx in skf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    temp_model = CatBoostClassifier(iterations=300, verbose=0)
    temp_model.fit(X_tr, y_tr, cat_features=cat_features)
    score = temp_model.score(X_val, y_val)
    cv_scores.append(score)

print(" CV Accuracy:", np.mean(cv_scores))

 CV Accuracy: 0.9413099870298314


In [12]:
# 11. PREDICTIONS

y_proba = model.predict_proba(X_test)[:, 1]


In [13]:
# 12. THRESHOLD TUNING

best_f1 = 0

for t in np.arange(0.1, 0.9, 0.05):
    preds = (y_proba > t).astype(int)
    score = f1_score(y_test, preds)
    if score > best_f1:
        best_f1 = score
        best_thresh = t

print(" Best Threshold:", best_thresh)
y_pred_optimized = (y_proba > best_thresh).astype(int)


 Best Threshold: 0.15000000000000002


In [14]:
# 13. CREATE MAIN OUTPUT FILE (IMPORTANT)

output_df = X_test.copy()

output_df['Actual'] = y_test.values
output_df['Predicted'] = y_pred_optimized
output_df['Fraud_Probability'] = y_proba
 
for feat in engineered_features:
    output_df[feat] = X_test[feat]

output_df.reset_index(drop=True, inplace=True)

# MAIN OUTPUT FILE
output_df.to_csv("../data/Tabular_model_output.csv", index=False)
print(" Main Output Saved: Tabular_model_output.csv")


 Main Output Saved: Tabular_model_output.csv


In [15]:
# 14. FEATURE FUSION READY FILE

fusion_df = output_df.copy()

fusion_columns = list(X.columns) + ['Fraud_Probability']
fusion_df = fusion_df[fusion_columns]

fusion_df.to_csv("../data/feature_fusion_input.csv", index=False)
print(" Feature Fusion File Saved: feature_fusion_input.csv")


 Feature Fusion File Saved: feature_fusion_input.csv


In [16]:
# 15. MODEL EVALUATION

print("\n Classification Report:")
print(classification_report(y_test, y_pred_optimized))

print("\n Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_optimized))

print("\n ROC-AUC:", roc_auc_score(y_test, y_proba))
print("\n PR-AUC:", average_precision_score(y_test, y_proba))



 Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.82      0.89      2899
           1       0.18      0.61      0.28       185

    accuracy                           0.81      3084
   macro avg       0.57      0.71      0.58      3084
weighted avg       0.92      0.81      0.85      3084


 Confusion Matrix:
[[2386  513]
 [  73  112]]

 ROC-AUC: 0.8234638225669615

 PR-AUC: 0.2110261555705591


In [17]:
# 16. FEATURE IMPORTANCE

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.get_feature_importance()
}).sort_values(by='Importance', ascending=False)

importance_df.to_csv("../data/feature_importance.csv", index=False)

print("\n Top Features:")
print(importance_df.head(10))



 Top Features:
                 Feature  Importance
16          driverrating    8.215491
20          ageofvehicle    7.672730
29            basepolicy    7.513436
1            weekofmonth    6.519090
32  hard_brakes_per_trip    6.409091
12            policytype    5.896100
11                 fault    5.881838
47    high_urban_driving    5.844651
21     ageofpolicyholder    5.406006
14          vehicleprice    5.072451
